# System setup

Create a tunnel to the DB on a secure office network or when connected via a VPN

Run :  ssh -L 5432:spiro-iot-ota-rdbms-prod.cj4o4wqe6xym.ap-south-1.rds.amazonaws.com:5432 spiro-nishnith-kumar@13.202.142.41 

then Run : nc -zv localhost 5432

if it says connection to localhost succeseeded! then go ahead with below code

In [7]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import List
from ota_loader import load_ota_data
import gspread
from gspread_dataframe import set_with_dataframe
import plotly.express as px
from datetime import datetime

# Code Files

In [11]:
trigger_data = load_ota_data("./ota/queries/ota_trigger.sql")
queue_data= load_ota_data("./ota/queries/ota_queue.sql")

/Users/Nishnith.B/Library/CloudStorage/OneDrive-EQUITANEDMCC/Documents/Codes/ota_loader.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


# Parameters

In [12]:
versions = ['1.4.3', 'CLY-6540B2.3']  
battery_family = ['U','C'] 
min_task_size = 500

# Data clean up

In [88]:
trigger_data['initiated_at'] = trigger_data['initiated_at'].astype(str)
trigger_data['completed_at'] = trigger_data['completed_at'].astype(str) 
queue_data['initiated_at'] = queue_data['initiated_at'].astype(str)
queue_data['completed_at'] = queue_data['completed_at'].astype(str)

trigger_data['initiated_at'] = trigger_data['initiated_at'].str.split('.').str[0]
trigger_data['completed_at'] = trigger_data['completed_at'].str.split('.').str[0]
queue_data['initiated_at'] = queue_data['initiated_at'].str.split('.').str[0]
queue_data['completed_at'] = queue_data['completed_at'].str.split('.').str[0]

trigger_data = trigger_data.dropna(subset=['batid'])  

In [89]:
df_copy = trigger_data.copy()

# Define configurable filters
def apply_filters(df, versions=None, battery_family=None):
    # Apply filters
    if versions is not None:
        df = df[(df['target_version'].isin(versions)) | (df['target_notes'].isin(versions))]
    
    if battery_family is not None:
        df = df[df['batid'].str.startswith(tuple(battery_family))]

    return df

filtered_df = apply_filters(df_copy, versions=versions, battery_family=battery_family)

In [90]:
analysis_df=filtered_df

In [91]:
task_sizes = analysis_df.groupby('task_id')['device_id'].nunique().reset_index()
task_sizes.columns = ['task_id', 'task_size']

valid_task_ids = task_sizes[task_sizes['task_size'] >= min_task_size]['task_id'].tolist()

# Filter the analysis_df to only include valid task_ids
final_df = analysis_df[analysis_df['task_id'].isin(valid_task_ids)]

# Overall OTA metrics

In [92]:
# Trigger Status Analysis
print("="*50)
print("TRIGGER STATUS ANALYSIS")
print("="*50)

# Total triggers sent
total_triggers = len(final_df)
print(f"\nTotal Triggers Sent: {total_triggers}")

# Get the breakdown of all statuses
print("\nStatus Breakdown:")
status_counts = final_df['status'].value_counts()
status_percentages = final_df['status'].value_counts(normalize=True) * 100

# Display the breakdown
print("-"*40)
for status in status_counts.index:
    count = status_counts[status]
    percentage = status_percentages[status]
    print(f"{status}: {count:,} ({percentage:.2f}%)")

# Specifically look for key statuses
print("\n" + "-"*40)
print("Key Status Categories:")
print("-"*40)

# Define the key statuses we're interested in
completed_count = final_df[final_df['status'] == 'completed'].shape[0] if 'completed' in final_df['status'].values else 0
in_progress_count = final_df[final_df['status'] == 'in_progress'].shape[0] if 'in_progress' in final_df['status'].values else 0
failed_count = final_df[final_df['status'] == 'failed'].shape[0] if 'failed' in final_df['status'].values else 0
offline_count = final_df[final_df['status'] == 'offline'].shape[0] if 'offline' in final_df['status'].values else 0

print(f"Completed: {completed_count:,} ({completed_count/total_triggers*100:.2f}%)")
print(f"In Progress: {in_progress_count:,} ({in_progress_count/total_triggers*100:.2f}%)")
print(f"Failed: {failed_count:,} ({failed_count/total_triggers*100:.2f}%)")
print(f"Offline: {offline_count:,} ({offline_count/total_triggers*100:.2f}%)")

# Success rate (considering completed as successful)
success_rate = (completed_count / total_triggers) * 100 if total_triggers > 0 else 0
print(f"\nOverall Success Rate: {success_rate:.2f}%")

# Create a summary DataFrame for visualization
summary_df = pd.DataFrame({
    'Status': ['Completed', 'In Progress', 'Failed', 'Offline', 'Other'],
    'Count': [
        completed_count,
        in_progress_count,
        failed_count,
        offline_count,
        total_triggers - (completed_count + in_progress_count + failed_count + offline_count)
    ]
})
summary_df['Percentage'] = (summary_df['Count'] / total_triggers) * 100
print("\n" + "="*50)
print("SUMMARY TABLE")
print("="*50)
print(summary_df.to_string(index=False))

TRIGGER STATUS ANALYSIS

Total Triggers Sent: 452084

Status Breakdown:
----------------------------------------
failed: 279,428 (61.81%)
offline: 132,349 (29.28%)
completed: 40,307 (8.92%)

----------------------------------------
Key Status Categories:
----------------------------------------
Completed: 40,307 (8.92%)
In Progress: 0 (0.00%)
Failed: 279,428 (61.81%)
Offline: 132,349 (29.28%)

Overall Success Rate: 8.92%

SUMMARY TABLE
     Status  Count  Percentage
  Completed  40307    8.915821
In Progress      0    0.000000
     Failed 279428   61.808867
    Offline 132349   29.275312
      Other      0    0.000000


In [93]:
# Task Level Analysis
print("="*50)
print("TASK LEVEL METRICS")
print("="*50)

# Calculate task-level statistics
task_metrics = final_df.groupby('task_id').agg({
    'device_id': 'nunique',  # task size (unique devices per task)
    'status': lambda x: {
        'total': len(x),
        'completed': (x == 'completed').sum(),
        'in_progress': (x == 'in_progress').sum(),
        'failed': (x == 'failed').sum(),
        'offline': (x == 'offline').sum()
    }
}).reset_index()

task_metrics.columns = ['task_id', 'task_size', 'status_breakdown']

# Extract status counts
task_metrics['total_triggers'] = task_metrics['status_breakdown'].apply(lambda x: x['total'])
task_metrics['completed_count'] = task_metrics['status_breakdown'].apply(lambda x: x['completed'])
task_metrics['in_progress_count'] = task_metrics['status_breakdown'].apply(lambda x: x['in_progress'])
task_metrics['failed_count'] = task_metrics['status_breakdown'].apply(lambda x: x['failed'])
task_metrics['offline_count'] = task_metrics['status_breakdown'].apply(lambda x: x['offline'])

# Calculate completion rate for each task
task_metrics['completion_rate'] = (task_metrics['completed_count'] / task_metrics['task_size']) * 100

# Basic Task Statistics
print(f"\nTotal Number of Tasks: {len(task_metrics)}")
print(f"Total Unique Devices Across All Tasks: {final_df['device_id'].nunique()}")

# Task Size Analysis
print("\n" + "-"*40)
print("TASK SIZE DISTRIBUTION")
print("-"*40)
print(f"Mean Task Size: {task_metrics['task_size'].mean():.2f}")
print(f"Median Task Size: {task_metrics['task_size'].median():.0f}")
print(f"Min Task Size: {task_metrics['task_size'].min()}")
print(f"Max Task Size: {task_metrics['task_size'].max()}")
print(f"Std Dev of Task Size: {task_metrics['task_size'].std():.2f}")

# Task size percentiles
print("\nTask Size Percentiles:")
percentiles = [10, 25, 50, 75, 90, 95, 99]
for p in percentiles:
    value = task_metrics['task_size'].quantile(p/100)
    print(f"  {p}th percentile: {value:.0f}")

# Task size buckets
print("\nTask Size Buckets:")
size_buckets = pd.cut(task_metrics['task_size'], 
                       bins=[0, 1, 5, 10, 50, 100, 500, 1000, float('inf')],
                       labels=['1', '2-5', '6-10', '11-50', '51-100', '101-500', '501-1000', '>1000'])
size_distribution = size_buckets.value_counts().sort_index()
print(size_distribution.to_string())

# Task Completion Analysis
print("\n" + "-"*40)
print("TASK COMPLETION METRICS")
print("-"*40)
print(f"Median Task Completion Rate: {task_metrics['completion_rate'].median():.2f}%")
print(f"Mean Task Completion Rate: {task_metrics['completion_rate'].mean():.2f}%")
print(f"Min Task Completion Rate: {task_metrics['completion_rate'].min():.2f}%")
print(f"Max Task Completion Rate: {task_metrics['completion_rate'].max():.2f}%")

# Tasks with different completion levels
fully_completed_tasks = (task_metrics['completion_rate'] == 100).sum()
partially_completed_tasks = ((task_metrics['completion_rate'] > 0) & (task_metrics['completion_rate'] < 100)).sum()
zero_completion_tasks = (task_metrics['completion_rate'] == 0).sum()

print(f"\nTasks with 100% Completion: {fully_completed_tasks} ({fully_completed_tasks/len(task_metrics)*100:.2f}%)")
print(f"Tasks with Partial Completion (0-100%): {partially_completed_tasks} ({partially_completed_tasks/len(task_metrics)*100:.2f}%)")
print(f"Tasks with 0% Completion: {zero_completion_tasks} ({zero_completion_tasks/len(task_metrics)*100:.2f}%)")

# Completion rate buckets
print("\nTask Completion Rate Distribution:")
completion_buckets = pd.cut(task_metrics['completion_rate'],
                            bins=[0, 10, 25, 50, 75, 90, 100],
                            labels=['0-10%', '10-25%', '25-50%', '50-75%', '75-90%', '90-100%'],
                            include_lowest=True)
completion_distribution = completion_buckets.value_counts().sort_index()
for bucket, count in completion_distribution.items():
    print(f"  {bucket}: {count} tasks ({count/len(task_metrics)*100:.2f}%)")

# Correlation between task size and completion rate
correlation = task_metrics['task_size'].corr(task_metrics['completion_rate'])
print(f"\nCorrelation between Task Size and Completion Rate: {correlation:.3f}")

TASK LEVEL METRICS

Total Number of Tasks: 309
Total Unique Devices Across All Tasks: 74125

----------------------------------------
TASK SIZE DISTRIBUTION
----------------------------------------
Mean Task Size: 1463.06
Median Task Size: 970
Min Task Size: 500
Max Task Size: 14987
Std Dev of Task Size: 1476.27

Task Size Percentiles:
  10th percentile: 527
  25th percentile: 557
  50th percentile: 970
  75th percentile: 1613
  90th percentile: 3554
  95th percentile: 4842
  99th percentile: 5670

Task Size Buckets:
task_size
1             0
2-5           0
6-10          0
11-50         0
51-100        0
101-500       3
501-1000    154
>1000       152

----------------------------------------
TASK COMPLETION METRICS
----------------------------------------
Median Task Completion Rate: 3.86%
Mean Task Completion Rate: 7.84%
Min Task Completion Rate: 0.00%
Max Task Completion Rate: 100.00%

Tasks with 100% Completion: 2 (0.65%)
Tasks with Partial Completion (0-100%): 298 (96.44%)
Tasks 

In [ ]:
# Step 1: Get first row per partition (device_id, target_notes) using id as timestamp proxy
def process_dataframes(final_df, queue_data):
    """
    Process final_df and queue_data according to requirements:
    1. Get first row per (device_id, target_notes) partition
    2. Filter for initiated_at > 2025-12-17
    3. Filter queue_data for completed_status == True and get first per group
    4. Left join the dataframes
    """
    
    print("Initial shapes:")
    print(f"final_df: {final_df.shape}")
    print(f"queue_data: {queue_data.shape}")
    
    # Step 1: Sort by id (proxy for timestamp) and get first row per partition
    print("\n1. Getting first row per (device_id, target_notes) partition...")
    
    # Sort by id to ensure we get the chronologically first row
    final_df_sorted = final_df.sort_values('id')
    
    # Get first row for each (device_id, target_notes) combination
    final_df_first = final_df_sorted.groupby(['device_id', 'target_notes'], as_index=False).first()
    
    print(f"After partitioning: {final_df_first.shape}")

    # Step 2: Filter for initiated_at after Dec 17, 2025
    print("\n2. Filtering for initiated_at > 2025-12-17...")
    
    # Convert initiated_at to datetime if it's not already
    final_df_first['initiated_at'] = pd.to_datetime(final_df_first['initiated_at'])
    cutoff_date = pd.to_datetime('2025-12-17')
    
    final_df_filtered = final_df_first[final_df_first['initiated_at'] > cutoff_date]
    
    print(f"After date filter: {final_df_filtered.shape}")
    if len(final_df_filtered) > 0:
        print(f"Date range: {final_df_filtered['initiated_at'].min()} to {final_df_filtered['initiated_at'].max()}")
    
    # Step 3: Filter queue_data for completed_status == True and get first per group
    print("\n3. Filtering queue_data for completed_status == True...")
    
    # Filter for True values only
    queue_data_true = queue_data[queue_data['completed_status'] == True].copy()
    
    print(f"Rows with completed_status == True: {queue_data_true.shape}")
    
    if len(queue_data_true) > 0:
        # Sort by id to get chronologically first True status
        queue_data_true = queue_data_true.sort_values('id')
        
        # Get first True completed_status for each (device_id, target_fw_id) combination
        queue_data_filtered = queue_data_true.groupby(['device_id', 'target_fw_id'], as_index=False).first()
        
        print(f"After getting first True per (device_id, target_fw_id): {queue_data_filtered.shape}")
    else:
        queue_data_filtered = queue_data_true
        print("No rows with completed_status == True found")
    
    # Step 4: Left join
    print("\n4. Performing left join...")
    
    # Rename queue_data columns to avoid conflicts (except join keys)
    queue_cols_renamed = {col: f'queue_{col}' for col in queue_data_filtered.columns 
                         if col not in ['device_id', 'target_fw_id']}
    queue_data_renamed = queue_data_filtered.rename(columns=queue_cols_renamed)
    
    # Perform left join
    result_df = final_df_filtered.merge(
        queue_data_renamed,
        left_on=['device_id', 'firmware_version_id'],
        right_on=['device_id', 'target_fw_id'],
        how='left',
        suffixes=('', '_queue_dup')  # In case of any remaining conflicts
    )
    
    # Drop the duplicate target_fw_id column if it exists
    if 'target_fw_id' in result_df.columns:
        result_df = result_df.drop('target_fw_id', axis=1)
    
    print(f"Final result shape: {result_df.shape}")
    
    return result_df, final_df_filtered, queue_data_filtered

# Validation function
def validate_results(original_final_df, original_queue_data, result_df, 
                    final_df_filtered, queue_data_filtered):
    """
    Comprehensive validation of the processing results
    """
    
    print("\n" + "="*50)
    print("VALIDATION RESULTS")
    print("="*50)
    
    # 1. Validate partitioning
    print("\n1. Partitioning Validation:")
    
    # Check for unique (device_id, target_notes) combinations
    partition_check = result_df.groupby(['device_id', 'target_notes']).size()
    duplicates = partition_check[partition_check > 1]
    
    if len(duplicates) == 0:
        print("✓ Each (device_id, target_notes) combination appears only once")
    else:
        print(f"✗ Found {len(duplicates)} duplicate partitions:")
        print(duplicates.head())
    
    # 2. Validate date filtering
    print("\n2. Date Filter Validation:")
    
    cutoff_date = pd.to_datetime('2025-12-17')
    invalid_dates = result_df[pd.to_datetime(result_df['initiated_at']) <= cutoff_date]
    
    if len(invalid_dates) == 0:
        print("✓ All rows have initiated_at > 2025-12-17")
        if len(result_df) > 0:
            print(f"  Earliest date: {result_df['initiated_at'].min()}")
    else:
        print(f"✗ Found {len(invalid_dates)} rows with initiated_at <= 2025-12-17")
    
    # 3. Validate queue_data filtering for True values
    print("\n3. Queue Data Filter Validation (completed_status == True):")
    
    if 'queue_completed_status' in result_df.columns:
        # Count True queue_completed_status (these are matched records)
        matched_with_queue = result_df['queue_completed_status'].notna().sum()
        
        # All matched records should have True completed_status
        true_status_count = (result_df['queue_completed_status'] == True).sum()
        
        print(f"✓ {matched_with_queue} rows matched with queue_data")
        print(f"  All {true_status_count} matched rows have completed_status = True")
        
        # Verify no False values
        false_count = (result_df['queue_completed_status'] == False).sum()
        if false_count == 0:
            print("✓ No rows with completed_status = False (as expected)")
        else:
            print(f"✗ Unexpected: Found {false_count} rows with completed_status = False")
    
    # 4. Statistics summary
    print("\n4. Processing Statistics:")
    print(f"Original final_df rows: {len(original_final_df)}")
    print(f"After partitioning: {len(final_df_filtered)}")
    print(f"Final result rows: {len(result_df)}")
    if len(original_final_df) > 0:
        print(f"Reduction ratio: {len(result_df)/len(original_final_df):.2%}")
    
    if 'queue_id' in result_df.columns:
        join_rate = result_df['queue_id'].notna().mean()
        print(f"Join success rate: {join_rate:.2%}")
    
    return result_df

# ============================================
# EXECUTE THE ANALYSIS
# ============================================

print("="*60)
print("STARTING DATA PROCESSING")
print("="*60)

# Process the dataframes
result_df, final_df_filtered, queue_data_filtered = process_dataframes(
    final_df.copy(), 
    queue_data.copy()
)

# Validate the results
final_result = validate_results(
    final_df, 
    queue_data, 
    result_df, 
    final_df_filtered, 
    queue_data_filtered
)

# ============================================
# SHOW SAMPLE OF RESULTS
# ============================================

print("\n" + "="*60)
print("SAMPLE OF FINAL RESULTS")
print("="*60)

if len(final_result) > 0:
    print("\nFirst 5 rows of processed data:")
    display_cols = ['device_id', 'target_notes', 'initiated_at', 'firmware_version_id']
    
    # Add queue columns if they exist
    queue_display_cols = ['queue_id', 'queue_completed_status', 'queue_completed_at']
    available_queue_cols = [col for col in queue_display_cols if col in final_result.columns]
    display_cols.extend(available_queue_cols)
    
    print(final_result[display_cols].head())
    
    print("\nSummary Statistics:")
    print(f"Total rows in result: {len(final_result)}")
    print(f"Unique devices: {final_result['device_id'].nunique()}")
    print(f"Unique target_notes: {final_result['target_notes'].nunique()}")
    
    if 'queue_id' in final_result.columns:
        matched_count = final_result['queue_id'].notna().sum()
        print(f"Rows matched with queue data: {matched_count} ({matched_count/len(final_result)*100:.1f}%)")
else:
    print("No data matches the criteria!")

# Save the result to a variable you can use
processed_data = final_result

print("\n" + "="*60)
print("PROCESSING COMPLETE!")
print("="*60)
print("\nThe processed data is now available in the 'processed_data' variable")
print("You can also access the intermediate results:")
print("  - result_df: The joined data")
print("  - final_df_filtered: Filtered final_df before join")
print("  - queue_data_filtered: Filtered queue_data before join")

STARTING DATA PROCESSING
Initial shapes:
final_df: (452084, 32)
queue_data: (73772, 11)

1. Getting first row per (device_id, target_notes) partition...
After partitioning: (74125, 32)

2. Filtering for initiated_at > 2025-12-17...
After date filter: (27985, 32)
Date range: 2025-12-17 12:28:20 to 2026-01-08 14:42:43

3. Filtering queue_data for completed_status == True...
Rows with completed_status == True: (45090, 11)
After getting first True per (device_id, target_fw_id): (45090, 11)

4. Performing left join...
Final result shape: (27985, 41)

VALIDATION RESULTS

1. Partitioning Validation:
✓ Each (device_id, target_notes) combination appears only once

2. Date Filter Validation:
✓ All rows have initiated_at > 2025-12-17
  Earliest date: 2025-12-17 12:28:20

3. Queue Data Filter Validation (completed_status == True):
✓ 14252 rows matched with queue_data
  All 14252 matched rows have completed_status = True
✓ No rows with completed_status = False (as expected)

4. Processing Statistic

In [99]:
import numpy as np
import pandas as pd

# Ensure datetime types (safe even if already datetime)
date_cols = [
    "initiated_at",
    "completed_at",
    "queue_completed_at"
]

for col in date_cols:
    processed_data[col] = pd.to_datetime(processed_data[col], errors="coerce")

# Compute time difference in days
processed_data["time_diff_days"] = np.where(
    processed_data["status"] == "completed",
    (processed_data["completed_at"] - processed_data["initiated_at"]).dt.days,
    (processed_data["queue_completed_at"] - processed_data["initiated_at"]).dt.days
)

# Round to whole number while preserving NaN
processed_data["time_diff_days"] = (
    processed_data["time_diff_days"]
    .astype("float")
    .round()
)


In [100]:
processed_data.head()

,device_id,target_notes,id,status,initiated_at,completed_at,message,firmware_version_id,task_id,requested_at,...,queue_id,queue_device_type,queue_message,queue_initiated_at,queue_task_id,queue_base_fw_version,queue_completed_at,queue_completed_status,queue_retry_count,time_diff_days
0,4.0,1.4.3,1071953,offline,2025-12-17 22:52:50,2025-12-17 22:52:50,offline,39,5912,2025-12-17 22:52:50.601911,...,755168.0,bms,None,2025-12-17 17:22:50,5912.0,1.3.0,2025-12-25 23:06:27,True,154.0,8.0
1,9.0,1.4.3,1082231,offline,2025-12-17 23:02:26,2025-12-17 23:02:26,offline,39,5921,2025-12-17 23:02:26.306732,...,764481.0,bms,None,2025-12-17 17:32:26,5921.0,,2025-12-19 02:54:20,True,24.0,1.0
2,15.0,1.4.3,996382,failed,2025-12-17 14:04:52,2025-12-17 14:04:52,"Falied,current: None: timestamp:None",39,5636,2025-12-17 14:04:52.419174,...,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN
3,19.0,1.4.3,1061319,failed,2025-12-17 20:32:41,2025-12-17 20:32:41,"Falied,current: None: timestamp:None",39,5847,2025-12-17 20:32:41.595178,...,747141.0,bms,None,2025-12-17 15:02:41,5847.0,1.3.0,2025-12-21 10:16:28,True,77.0,3.0
4,22.0,1.4.3,1070980,failed,2025-12-17 22:52:46,2025-12-17 23:14:44,Timeout — no response for 20 minutes,39,5912,2025-12-17 22:52:46.425631,...,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN


In [101]:
date_cols = [
    "initiated_at",
    "completed_at",
    "queue_completed_at"
]

for col in date_cols:
    processed_data[col] = pd.to_datetime(processed_data[col], errors="coerce")


In [102]:
processed_data["time_diff_days"] = np.where(
    processed_data["status"] == "completed",
    (processed_data["completed_at"] - processed_data["initiated_at"]).dt.days,
    (processed_data["queue_completed_at"] - processed_data["initiated_at"]).dt.days
)

processed_data["time_diff_days"] = (
    processed_data["time_diff_days"]
    .astype("float")
    .round()
)


In [103]:
metrics = {
    "total_unique_devices": processed_data["device_id"].nunique(),

    "completed_devices": processed_data.loc[
        processed_data["status"] == "completed", "device_id"
    ].nunique(),

    "offline_devices": processed_data.loc[
        processed_data["status"] == "offline", "device_id"
    ].nunique(),

    "failed_devices": processed_data.loc[
        processed_data["status"] == "failed", "device_id"
    ].nunique(),

    "offline_with_queue_completed_true": processed_data.loc[
        (processed_data["status"] == "offline") &
        (processed_data["queue_completed_status"] == True),
        "device_id"
    ].nunique(),

    "failed_with_queue_completed_true": processed_data.loc[
        (processed_data["status"] == "failed") &
        (processed_data["queue_completed_status"] == True),
        "device_id"
    ].nunique()
}

metrics


{'total_unique_devices': 27985,
 'completed_devices': 3150,
 'offline_devices': 12176,
 'failed_devices': 12659,
 'offline_with_queue_completed_true': 3077,
 'failed_with_queue_completed_true': 10576}

In [104]:
processed_data["batid_group"] = (
    processed_data["batid"]
    .astype(str)
    .str[0]
    .fillna("Unknown")
)


In [105]:
plot_df = processed_data[
    processed_data["time_diff_days"].notna() &
    processed_data["device_id"].notna()
].copy()


In [106]:
processed_data["time_diff_hours"] = np.where(
    processed_data["status"] == "completed",
    (processed_data["completed_at"] - processed_data["initiated_at"]).dt.total_seconds() / 3600,
    (processed_data["queue_completed_at"] - processed_data["initiated_at"]).dt.total_seconds() / 3600
)

processed_data["time_diff_hours"] = (
    processed_data["time_diff_hours"]
    .astype("float")
    .round()
)


In [107]:
hourly_df = processed_data[
    processed_data["time_diff_hours"].notna() &
    (processed_data["time_diff_hours"] >= 0) &
    (processed_data["time_diff_hours"] <= 96) &
    processed_data["device_id"].notna()
].copy()


In [108]:
processed_data["batid_group"] = (
    processed_data["batid"]
    .astype(str)
    .str[0]
    .fillna("Unknown")
)


In [109]:
batid_totals = (
    processed_data
    .groupby("batid_group")["device_id"]
    .nunique()
    .reset_index(name="total_devices_batid")
)


In [110]:
plot_df = processed_data[
    processed_data["time_diff_days"].notna() &
    processed_data["device_id"].notna()
].copy()


In [111]:
agg_df = (
    plot_df
    .groupby(["batid_group", "time_diff_days"])["device_id"]
    .nunique()
    .reset_index(name="daily_unique_devices")
    .sort_values(["batid_group", "time_diff_days"])
)


In [112]:
agg_df["cumulative_devices"] = (
    agg_df
    .groupby("batid_group")["daily_unique_devices"]
    .cumsum()
)


In [113]:
agg_df = agg_df.merge(
    batid_totals,
    on="batid_group",
    how="left"
)

agg_df["cumulative_percent"] = (
    agg_df["cumulative_devices"]
    / agg_df["total_devices_batid"]
) * 100


In [114]:
import plotly.express as px

fig = px.line(
    agg_df,
    x="time_diff_days",
    y="cumulative_percent",
    color="batid_group",
    markers=True,
    title="OTA Progress by BATID Group (% of Total Devices)",
    labels={
        "time_diff_days": "Time Delta (Days)",
        "cumulative_percent": "Cumulative Devices (%)",
        "batid_group": "BATID First Letter"
    }
)

fig.update_layout(
    template="plotly_dark",
    hovermode="x unified",
    yaxis=dict(ticksuffix="%")
)

fig.show()


In [115]:
import numpy as np
import pandas as pd

processed_data["time_diff_hours"] = np.where(
    processed_data["status"] == "completed",
    (processed_data["completed_at"] - processed_data["initiated_at"]).dt.total_seconds() / 3600,
    (processed_data["queue_completed_at"] - processed_data["initiated_at"]).dt.total_seconds() / 3600
)

processed_data["time_diff_hours"] = (
    processed_data["time_diff_hours"]
    .astype("float")
    .round()
)


In [116]:
processed_data["batid_group"] = (
    processed_data["batid"]
    .astype(str)
    .str[0]
    .fillna("Unknown")
)


In [117]:
batid_totals = (
    processed_data
    .groupby("batid_group")["device_id"]
    .nunique()
    .reset_index(name="total_devices_batid")
)


In [118]:
plot_df_hourly = processed_data[
    processed_data["time_diff_hours"].notna() &
    (processed_data["time_diff_hours"] >= 0) &
    (processed_data["time_diff_hours"] <= 96) &
    processed_data["device_id"].notna()
].copy()


In [119]:
hourly_agg = (
    plot_df_hourly
    .groupby(["batid_group", "time_diff_hours"])["device_id"]
    .nunique()
    .reset_index(name="hourly_unique_devices")
    .sort_values(["batid_group", "time_diff_hours"])
)


In [120]:
hourly_agg["cumulative_devices"] = (
    hourly_agg
    .groupby("batid_group")["hourly_unique_devices"]
    .cumsum()
)


In [122]:
hourly_agg = hourly_agg.merge(
    batid_totals,
    on="batid_group",
    how="left"
)

hourly_agg["cumulative_percent"] = (
    hourly_agg["cumulative_devices"]
    / hourly_agg["total_devices_batid"]
) * 100


KeyError: 'total_devices_batid'

In [123]:
import plotly.express as px

fig_hourly = px.line(
    hourly_agg,
    x="time_diff_hours",
    y="cumulative_percent",
    color="batid_group",
    markers=True,
    title="Hourly OTA Progress by BATID Group (% of Total Devices, First 4 Days)",
    labels={
        "time_diff_hours": "Time Delta (Hours)",
        "cumulative_percent": "Cumulative Devices (%)",
        "batid_group": "BATID First Letter"
    }
)

fig_hourly.update_layout(
    template="plotly_dark",
    hovermode="x unified",
    yaxis=dict(ticksuffix="%"),
    xaxis=dict(
        tickmode="linear",
        dtick=6  # change to 1 for every-hour ticks
    )
)

fig_hourly.show()
